# Metadata Prediction Metrics

This notebook loads completed rows from `metadata_predictions`, computes task-level metrics, and reports how many rows were usable vs empty for each metadata task.

In [1]:
import json
import sys
from pathlib import Path

project_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "src" / "data_collection" / "consts.py").is_file()
    ),
    None,
)
if project_root is None:
    raise RuntimeError("Could not locate project root containing 'src' directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import psycopg2
from IPython.display import display
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from src.data_collection.consts import DB_PARAMS

In [2]:
TASK_CONFIGS = {
    "report_type": {
        "true_col": "true_report_type",
        "pred_col": "pred_report_type",
        "confidence_col": "report_type_confidence",
        "probabilities_col": "report_type_probabilities",
    },
    "company_quarter": {
        "true_col": "true_company_quarter",
        "pred_col": "pred_company_quarter",
        "confidence_col": "company_quarter_confidence",
        "probabilities_col": "company_quarter_probabilities",
    },
    "sector": {
        "true_col": "true_sector",
        "pred_col": "pred_sector",
        "confidence_col": "sector_confidence",
        "probabilities_col": "sector_probabilities",
    },
    "market_cap": {
        "true_col": "true_market_cap",
        "pred_col": "pred_market_cap",
        "confidence_col": "market_cap_confidence",
        "probabilities_col": "market_cap_probabilities",
    },
}

ALL_REQUIRED_COLUMNS = sorted(
    {
        col
        for config in TASK_CONFIGS.values()
        for col in [
            config["true_col"],
            config["pred_col"],
            config["confidence_col"],
            config["probabilities_col"],
        ]
    }
)


def load_predictions() -> pd.DataFrame:
    query = "SELECT * FROM metadata_predictions ORDER BY report_id"
    with psycopg2.connect(**DB_PARAMS) as conn:
        return pd.read_sql_query(query, conn)


def normalize_probability_dict(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, dict):
        return value
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return None
    if hasattr(value, "items"):
        return dict(value)
    return None


def probability_for_label(probabilities, label):
    prob_dict = normalize_probability_dict(probabilities)
    if not prob_dict or pd.isna(label):
        return None

    candidates = []
    if isinstance(label, float) and float(label).is_integer():
        label_int = int(label)
        candidates.extend([label, label_int, str(label), str(label_int)])
    elif isinstance(label, int):
        candidates.extend([label, float(label), str(label)])
    elif isinstance(label, str):
        stripped = label.strip()
        candidates.extend([label, stripped])
        try:
            parsed_float = float(stripped)
            candidates.append(parsed_float)
            if parsed_float.is_integer():
                parsed_int = int(parsed_float)
                candidates.extend([parsed_int, str(parsed_int)])
        except ValueError:
            pass
    else:
        candidates.extend([label, str(label)])

    seen = set()
    unique_candidates = []
    for candidate in candidates:
        marker = repr(candidate)
        if marker not in seen:
            seen.add(marker)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate in prob_dict:
            value = prob_dict[candidate]
            if value is None:
                return None
            return float(value)
    return None


predictions_df = load_predictions()
total_rows = len(predictions_df)
fully_completed_rows = int(predictions_df[ALL_REQUIRED_COLUMNS].notna().all(axis=1).sum()) if total_rows else 0

print(f"Total rows in metadata_predictions: {total_rows}")
print(f"Fully completed rows across all four tasks: {fully_completed_rows}")
print(f"Rows not fully completed: {total_rows - fully_completed_rows}")

preview_columns = [
    col for col in predictions_df.columns if not col.endswith("_probabilities")
]
display(predictions_df[preview_columns])


Total rows in metadata_predictions: 3300
Fully completed rows across all four tasks: 3272
Rows not fully completed: 28


/tmp/ipykernel_7810/2691464997.py:45: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  return pd.read_sql_query(query, conn)


,report_id,true_report_type,pred_report_type,report_type_confidence,true_company_quarter,pred_company_quarter,company_quarter_confidence,true_sector,pred_sector,sector_confidence,true_market_cap,pred_market_cap,market_cap_confidence
0,4,10-K,10-K,0.213558,4,4,0.108558,Healthcare,Healthcare,0.102728,mid,mid,0.022084
1,9,10-K,10-K,0.442749,4,4,0.080883,Healthcare,Healthcare,0.189457,large,small,0.011261
2,14,10-Q,10-Q,0.258171,2,4,0.057407,Consumer Defensive,Consumer Cyclical,0.097697,large,small,0.086043
3,21,10-Q,10-Q,0.452530,3,1,0.191723,Technology,Technology,0.350608,large,mid,0.068158
4,24,10-K,10-K,0.189720,4,4,0.070818,Utilities,Utilities,0.575042,large,small,0.030662
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3295,31530,10-Q,10-Q,0.509857,2,2,0.175208,Healthcare,Healthcare,0.162810,large,large,0.011268
3296,34228,10-Q,10-Q,0.385986,1,3,0.086122,Consumer Defensive,Consumer Defensive,0.436709,mid,small,0.171532
3297,34682,10-Q,10-Q,0.452042,1,3,0.104199,Consumer Defensive,Consumer Defensive,0.414092,large,mid,0.119824
3298,35017,10-Q,10-Q,0.278557,3,3,0.046910,Consumer Defensive,Consumer Defensive,0.360888,mid,mid,0.116590


In [3]:
def evaluate_task(df: pd.DataFrame, task_name: str, config: dict) -> tuple[dict, pd.DataFrame]:
    required_cols = [
        config["true_col"],
        config["pred_col"],
        config["confidence_col"],
        config["probabilities_col"],
    ]
    task_df = df[["report_id", *required_cols]].copy()
    used_mask = task_df[required_cols].notna().all(axis=1)
    used_df = task_df[used_mask].copy()

    rows_used = int(used_mask.sum())
    rows_empty = int((~used_mask).sum())

    if used_df.empty:
        return {
            "task": task_name,
            "rows_used": rows_used,
            "rows_empty": rows_empty,
            "rows_missing_true_label_probability": 0,
            "avg_confidence": None,
            "avg_true_label_probability": None,
            "accuracy": None,
            "precision_macro": None,
            "recall_macro": None,
            "f1_macro": None,
        }, used_df

    true_col = config["true_col"]
    pred_col = config["pred_col"]
    conf_col = config["confidence_col"]
    prob_col = config["probabilities_col"]

    if task_name == "company_quarter":
        used_df[true_col] = used_df[true_col].astype(int)
        used_df[pred_col] = used_df[pred_col].astype(int)

    used_df["true_label_probability"] = used_df.apply(
        lambda row: probability_for_label(row[prob_col], row[true_col]),
        axis=1,
    )

    y_true = used_df[true_col]
    y_pred = used_df[pred_col]

    metrics = {
        "task": task_name,
        "rows_used": rows_used,
        "rows_empty": rows_empty,
        "rows_missing_true_label_probability": int(used_df["true_label_probability"].isna().sum()),
        "avg_confidence": used_df[conf_col].mean(),
        "avg_true_label_probability": used_df["true_label_probability"].mean(),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
    }

    return metrics, used_df


all_metrics = []
task_frames = {}

for task_name, config in TASK_CONFIGS.items():
    metrics, task_df = evaluate_task(predictions_df, task_name, config)
    all_metrics.append(metrics)
    task_frames[task_name] = task_df

results_df = pd.DataFrame(all_metrics)
display(results_df)


,task,rows_used,rows_empty,rows_missing_true_label_probability,avg_confidence,avg_true_label_probability,accuracy,precision_macro,recall_macro,f1_macro
0,report_type,3300,0,0,0.406814,0.728779,0.935758,0.948457,0.889514,0.913641
1,company_quarter,3300,0,0,0.076318,0.445572,0.773939,0.770894,0.769795,0.763007
2,sector,3286,14,0,0.286688,0.516154,0.735849,0.753685,0.718453,0.689907
3,market_cap,3272,28,0,0.066622,0.320076,0.299205,0.351627,0.368821,0.244351


In [4]:
for task_name, task_df in task_frames.items():
    print(f"\n{task_name}")
    print(f"rows used: {len(task_df)}")
    if task_df.empty:
        print("No completed rows for this task.")
        continue

    if task_name == "company_quarter":
        display(
            task_df[[
                "report_id",
                "true_company_quarter",
                "pred_company_quarter",
                "company_quarter_confidence",
                "true_label_probability",
            ]].head()
        )
        display(pd.crosstab(task_df["true_company_quarter"], task_df["pred_company_quarter"]))
    else:
        cfg = TASK_CONFIGS[task_name]
        display(
            task_df[[
                "report_id",
                cfg["true_col"],
                cfg["pred_col"],
                cfg["confidence_col"],
                "true_label_probability",
            ]].head()
        )
        display(pd.crosstab(task_df[cfg["true_col"]], task_df[cfg["pred_col"]]))



report_type
rows used: 3300


,report_id,true_report_type,pred_report_type,report_type_confidence,true_label_probability
0,4,10-K,10-K,0.213558,0.652412
1,9,10-K,10-K,0.442749,0.519438
2,14,10-Q,10-Q,0.258171,0.796359
3,21,10-Q,10-Q,0.452530,0.840476
4,24,10-K,10-K,0.189720,0.627177


pred_report_type,10-K,10-Q
true_report_type,,
10-K,709,191
10-Q,21,2379



company_quarter
rows used: 3300


,report_id,true_company_quarter,pred_company_quarter,company_quarter_confidence,true_label_probability
0,4,4,4,0.108558,0.544437
1,9,4,4,0.080883,0.447560
2,14,2,4,0.057407,0.051244
3,21,3,1,0.191723,0.136187
4,24,4,4,0.070818,0.399478


...

,report_id,true_sector,pred_sector,sector_confidence,true_label_probability
0,4,Healthcare,Healthcare,0.102728,0.573944
1,9,Healthcare,Healthcare,0.189457,0.261271
2,14,Consumer Defensive,Consumer Cyclical,0.097697,0.093889
3,21,Technology,Technology,0.350608,0.865022
4,24,Utilities,Utilities,0.575042,0.469247


pred_sector,Basic Materials,Communication Services,Consumer Cyclical,Consumer Defensive,Energy,Financial Services,Healthcare,Industrials,Real Estate,Technology,Utilities
true_sector,,,,,,,,,,,
Basic Materials,94,0,0,9,14,0,0,11,0,0,0
Communication Services,0,102,0,0,0,1,0,0,0,21,0
Consumer Cyclical,17,48,116,50,4,50,1,50,0,31,0
Consumer Defensive,3,0,71,163,0,1,1,0,0,0,0
Energy,0,0,0,0,147,0,0,0,0,0,0
Financial Services,0,0,0,0,4,451,0,0,0,0,0
Healthcare,1,0,2,0,0,19,350,8,0,13,0
Industrials,20,0,2,0,31,38,0,347,0,29,2
Real Estate,6,24,2,0,2,110,9,10,37,17,0



market_cap
rows used: 3272


,report_id,true_market_cap,pred_market_cap,market_cap_confidence,true_label_probability
0,4,mid,mid,0.022084,0.470940
1,9,large,small,0.011261,0.270241
2,14,large,small,0.086043,0.234302
3,21,large,mid,0.068158,0.437363
4,24,large,small,0.030662,0.321400


pred_market_cap,large,mid,small
true_market_cap,,,
large,565,1234,639
mid,134,388,241
small,15,30,26
